# TP : Web Sémantique avec RDFLib et SPARQL

### PARTIE 1. Chargeons nos données

In [80]:
# rdflib est un équivalent de jena en Java, SPARQLWrapper permet d'utiliser les endpoints publiques
!pip install rdflib SPARQLWrapper

Defaulting to user installation because normal site-packages is not writeable


Vous est fourni university.n3, lisez le et comprenez son contenu avant de continuer.

In [9]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL, URIRef
from rdflib.namespace import XSD

# Création du graphe
g = Graph()

# Chargement du fichier N3
g.parse("./university.n3", format="n3", encoding="utf-8")

# Affichage du nombre de triplets
print(f"Nombre de triplets dans le graphe : {len(g)}")


print(f"\n✓ Base de connaissances créée avec {len(g)} triplets RDF")
print("\nExemple de quelques triplets :")
for i, (s, p, o) in enumerate(g):
    if i < 5:
        print(f"  {s} -> {p} -> {o}")

Nombre de triplets dans le graphe : 60

✓ Base de connaissances créée avec 60 triplets RDF

Exemple de quelques triplets :
  http://example.org/university#Student -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://www.w3.org/2000/01/rdf-schema#Class
  http://example.org/university#Prof_Bob -> http://example.org/university#belongsTo -> http://example.org/university#ComputerScience
  http://example.org/university#MachineLearning -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://example.org/university#Course
  http://example.org/university#Student_Diana -> http://example.org/university#hasAge -> 21
  http://example.org/university#hasAge -> http://www.w3.org/2000/01/rdf-schema#range -> http://www.w3.org/2001/XMLSchema#integer


EXERCICE 1 :

⛳ TODO Enrichir la base de connaissances
- Créez un nouveau département "Mathematics"
- Ajoutez un professeur au département de mathématiques
- Ajoutez 2 nouveaux étudiants avec leurs cours

In [ ]:
# Nouveau département Mathematics
ex:Mathematics a rdfs:Department;
    rdfs:label "Mathematics" .

# Nouveau professeur de mathématiques
ex:Zani a ex:Professor;
    foaf:name "Marguerite Zani";
    ex:hasAge "52";
    ex:belongsTo ex:Mathematics;
    ex:teaches ex:Stats .
    
# Nouveau cours de mathématiques
ex:Stats a ex:Course ;
    rdfs:label "Stats"
    
# Deux nouveaux étudiants
ex:Benjamin a ex:Student ;
    foaf:name "Benjamin Perez" ;
    ex:hasAge "21" ;
    ex:attends ex:Stats .
    
ex:Vincent a ex:Student ;
    foaf:name "Vincent Gonnet" ;
    ex:hasAge "23" ;
    ex:attends ex:Stats .

### PARTIE 2 : Requêtes SPARQL locales

Nous allons procéder à plusieurs requêtes SPARQL afin de :
*   Lister tous les étudiants
*   Donner la liste des cours enseignés par chaque professeur
*   Donner la liste des étudiants et les cours qu'ils suivent
*   Construire un graphe contenant les relations étudiant - professeur tel que si un étudiant suit un cours (*attends*) et que qu'un professeur enseigne ce cours (*teaches*) alors l'étudiant à pour enseignant ce professeur

In [10]:
# Requête 1 : Lister tous les étudiants
print("\n--- Requête 1 : Tous les étudiants ---")
query1 = """
PREFIX ex: <http://example.org/university#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?student ?name
WHERE {
    ?student rdf:type ex:Student .
    ?student foaf:name ?name .
}
"""
results1 = g.query(query1)
for row in results1:
    print(f"  Étudiant: {row.name}")


--- Requête 1 : Tous les étudiants ---
  Étudiant: Charlie Durand
  Étudiant: Diana Leclerc
  Étudiant: Benjamin Perez
  Étudiant: Vincent Gonnet


In [11]:
# Requête 2 : Cours enseignés par chaque professeur
print("\n--- Requête 2 : Cours par professeur ---")
query2 = """
PREFIX ex: <http://example.org/university#>

SELECT ?profName ?courseName
WHERE {
    ?prof rdf:type ex:Professor .
    ?prof foaf:name ?profName .
    ?prof ex:teaches ?course .
    ?course rdfs:label ?courseName .
}
"""
results2 = g.query(query2)
for row in results2:
    print(f"  {row.profName} enseigne {row.courseName}")


--- Requête 2 : Cours par professeur ---
  Alice Dupont enseigne Web Semantique
  Bob Martin enseigne Machine Learning
  Marguerite Zani enseigne Stats


In [12]:
# Requête 3 : Étudiants et leurs cours
print("\n--- Requête 3 : Étudiants et leurs cours ---")
query3 = """
PREFIX ex: <http://example.org/university#>

SELECT ?studentName ?courseName
WHERE {
    ?student rdf:type ex:Student .
    ?student foaf:name ?studentName .
    ?student ex:attends ?course .
    ?course rdfs:label ?courseName .
}
ORDER BY ?studentName
"""
results3 = g.query(query3)
for row in results3:
    print(f"  {row.studentName} suit {row.courseName}")


--- Requête 3 : Étudiants et leurs cours ---
  Benjamin Perez suit Stats
  Charlie Durand suit Web Semantique
  Diana Leclerc suit Web Semantique
  Diana Leclerc suit Machine Learning
  Vincent Gonnet suit Stats


In [13]:
# Requête 4 : CONSTRUCT - Créer de nouveaux triplets
print("\n--- Requête 4 : CONSTRUCT - Relations étudiant-professeur ---")
query4 = """
PREFIX ex: <http://example.org/university#>

CONSTRUCT {
    ?student ex:hasTeacher ?prof .
}
WHERE {
    ?student ex:attends ?course .
    ?prof ex:teaches ?course .
}
"""
new_graph = g.query(query4).graph
print(f"  {len(new_graph)} nouveaux triplets créés")
for s, p, o in new_graph:
    print(f"  {s.split('#')[1]} -> hasTeacher -> {o.split('#')[1]}")


--- Requête 4 : CONSTRUCT - Relations étudiant-professeur ---
  5 nouveaux triplets créés
  Student_Charlie -> hasTeacher -> Prof_Alice
  Student_Diana -> hasTeacher -> Prof_Bob
  Benjamin -> hasTeacher -> Zani
  Vincent -> hasTeacher -> Zani
  Student_Diana -> hasTeacher -> Prof_Alice


EXERCICE 2 :

⛳ TODO Requêtes SPARQL 
- Écrire une requête pour trouver les étudiants qui suivent plus d'un cours
- Créer une requête CONSTRUCT pour inférer les collègues (profs du même département)
- Utiliser OPTIONAL pour afficher tous les professeurs et leurs départements (si disponible)




In [65]:
# ⛳ TODO
# Liste des étudiants qui suivent plus d'un cours
query5 = """
PREFIX ex: <http://example.org/university#>

SELECT ?studentName
WHERE {
    ?student rdf:type ex:Student .
    ?student foaf:name ?studentName .
    ?student ex:attends ?course .
}
GROUP BY ?studentName
HAVING (COUNT(?course) > 1)
"""
res4 = g.query(query5)
for row in res4:
    print(f"  {row.studentName} ")

  Diana Leclerc 


In [85]:
# ⛳ TODO
# Les collègues avec CONSTRUCT
query5 = """
PREFIX ex: <http://example.org/university#>

CONSTRUCT {
    ?prof1 ex:colleagueOf ?prof2 .
}
WHERE {
    ?prof1 rdf:type ex:Professor .
    ?prof2 rdf:type ex:Professor .
    ?prof1 ex:belongsTo ?dept .
    ?prof2 ex:belongsTo ?dept .
    FILTER(?prof1 != ?prof2)
}
"""
res5 = g.query(query5).graph
for s, p, o in res5:
    print(f"  {s.split('#')[1]} -> isCollegue -> {o.split('#')[1]}")

  Prof_Alice -> isCollegue -> Prof_Bob
  Prof_Bob -> isCollegue -> Prof_Alice


In [94]:
# ⛳ TODO
# Profs et Depts avec OPTIONAL
query6="""
PREFIX ex: <http://example.org/university#>

SELECT ?name ?deptName
WHERE {
    ?prof rdf:type ex:Professor .
    ?prof foaf:name ?name .
    OPTIONAL {
        ?prof ex:belongsTo ?dept .
        ?dept rdfs:label ?deptName
    }
}
"""
res6 = g.query(query6)
for row in res6:
    print(f"  {row.name} {row.deptName}")

  Alice Dupont Computer Science
  Bob Martin Computer Science
  Marguerite Zani Mathematique


### PARTIE 3 : Raisonnement avec inférences RDFS

In [ ]:
!pip install owlrl

In [ ]:
#from rdflib.plugins.stores.memory import Memory
#from rdflib.plugins.reasoners import RDFS # This module is deprecated and not available by default
import owlrl

# Créer un graphe pour le raisonnement (copie du graphe original)
reasoning_graph = Graph()
reasoning_graph.parse(data=g.serialize(format='turtle'), format='turtle')

# Ajouter une règle d'inférence personnalisée (exemple conceptuel, non implémenté directement ici)
# Si quelqu'un enseigne un cours, il est membre du département qui offre ce cours
print("\n--- Avant le raisonnement ---")
query_persons = """
PREFIX ex: <http://example.org/university#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?person
WHERE {
    ?person rdf:type foaf:Person .
}
"""
results_before = reasoning_graph.query(query_persons)
print(f"Nombre de personnes trouvées : {len(list(results_before))}")

In [ ]:
# Compter les triplets avant
triplets_avant = len(reasoning_graph)

# Appliquer le raisonnement RDFS automatiquement
owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(reasoning_graph)

# Compter après
triplets_apres = len(reasoning_graph)
inferred_count = triplets_apres - triplets_avant

print(f"✓ Raisonnement RDFS appliqué automatiquement")
print(f"  Triplets avant : {triplets_avant}")
print(f"  Triplets après : {triplets_apres}")
print(f"  Triplets inférés : {inferred_count}")

print("\n  Inférences effectuées par OWL :")
print("  • rdfs:subClassOf → propagation des types")
print("  • rdfs:domain/range → typage automatique")
print("  • rdfs:subPropertyOf → héritage de propriétés")


In [ ]:
print("\n--- Après le raisonnement ---")
query_persons = """
PREFIX ex: <http://example.org/university#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?person
WHERE {
    ?person rdf:type ex:Person .
}
"""
results_after = reasoning_graph.query(query_persons)
print(f"Nombre de personnes trouvées : {len(list(results_after))}")

In [ ]:
persons = list(results_after)
print(f"\nNombre de personnes trouvées : {len(persons)}")
print("Personnes identifiées :")
for individu in persons:
    name_query = f"""
    PREFIX ex: <http://example.org/university#>
    SELECT ?name WHERE {{ <{individu.person}> ex:hasName ?name }}
    """
    name_results = list(reasoning_graph.query(name_query))
    if name_results:
        print(f"  - {name_results[0].name}")

### PARTIE 4 : Interrogation d'un endpoint SPARQL réel 

In [ ]:
!pip install SPARQLWrapper

Nous allons interroger DbPedia pour des requêtes simples :

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

# Configuration de l'endpoint DBpedia
sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)

In [ ]:
# Requête 1 : Trouver des informations sur Paris
print("\n--- Requête 1 : Informations sur Paris ---")
query_paris = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?property ?value
WHERE {
    dbr:Paris ?property ?value .
    FILTER(LANG(?value) = "fr" || LANG(?value) = "")
}
LIMIT 10
"""
sparql.setQuery(query_paris)

try:
    results = sparql.query().convert()
    print("Propriétés de Paris (extrait) :")
    for result in results["results"]["bindings"][:5]:
        prop = result["property"]["value"].split('/')[-1]
        value = result["value"]["value"]
        if len(value) < 100:
            print(f"  {prop}: {value[:80]}")
except Exception as e:
    print(f"  Erreur : {e}")



In [ ]:
# Requête 2 : Universités françaises
print("\n--- Requête 2 : Universités françaises ---")
query_universities = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?uni ?name
WHERE {
    ?uni a dbo:University ;
         dbo:country dbr:France ;
         rdfs:label ?name .
    FILTER(LANG(?name) = "fr")
}
LIMIT 10
"""
sparql.setQuery(query_universities)

try:
    results = sparql.query().convert()
    print("Quelques universités françaises :")
    for result in results["results"]["bindings"]:
        print(f"  - {result['name']['value']}")
except Exception as e:
    print(f"  Erreur : {e}")


EXERCICE 3 :

⛳ TODO Exploration de DBpedia
- Trouvez les films réalisés par un réalisateur français de votre choix
- Listez les fleuves de plus de 1000 km de long


In [ ]:
# ⛳ TODO
# Films d'un réalisateur


In [ ]:
# ⛳ TODO
# Fleuves > 1000 km de long
